In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

print("=== FINSIGHT AI: STEP 2 - REAL-WORLD PREPROCESSING & HP FILTER ===\n")

# ---------------------------------------------------------
# 1. LOAD CLEANED RAW DATA
# ---------------------------------------------------------
try:
    df = pd.read_csv('data/01_cleaned_raw.csv')
    print(f"Successfully loaded data: {df.shape[0]} rows.")
except FileNotFoundError:
    print("❌ ERROR: 01_cleaned_raw.csv not found in the 'data/' folder.")
    raise

# Sort by country and year to maintain chronological order
if 'year' in df.columns and 'country' in df.columns:
    df = df.sort_values(by=['country', 'year']).reset_index(drop=True)

# ---------------------------------------------------------
# 2. FEATURE ENGINEERING (Yield Curve & Credit Proxies)
# ---------------------------------------------------------
# Calculate yield curve slope if ltrate and stir exist
if 'ltrate' in df.columns and 'stir' in df.columns:
    df['yield_curve_slope'] = df['ltrate'] - df['stir']

# Map tloans as credit metric if credit_gdp isn't explicitly named
if 'tloans' in df.columns and 'gdp' in df.columns:
    df['credit_gdp'] = df['tloans'] / df['gdp']
elif 'tloans' in df.columns:
    df['credit_gdp'] = df['tloans']

# Calculate second difference for credit momentum
if 'credit_gdp' in df.columns:
    df['credit_gdp_diff2'] = df.groupby('country')['credit_gdp'].diff().diff().fillna(0)
else:
    df['credit_gdp_diff2'] = 0

# ---------------------------------------------------------
# 3. APPLY THE WINNING HP (HODRICK-PRESCOTT) FILTER
# ---------------------------------------------------------
def apply_hp_filter(series, lamb=100):
    """
    Applies the Hodrick-Prescott (HP) Filter to isolate cyclical fluctuations.
    lamb=100 is the standard smoothing parameter for annual macroeconomic series.
    """
    valid_series = series.dropna()
    if len(valid_series) < 10:
        return pd.Series(0, index=series.index)
    
    try:
        cycle, _ = sm.tsa.filters.hpfilter(valid_series, lamb=lamb)
        out = pd.Series(np.nan, index=series.index)
        out.loc[cycle.index] = cycle
        return out.fillna(0)
    except Exception:
        return pd.Series(0, index=series.index)

print("Applying the winning HP Filter grouped by country...")

if 'credit_gdp' in df.columns:
    df['credit_gdp_cycle'] = df.groupby('country')['credit_gdp'].transform(apply_hp_filter)

if 'yield_curve_slope' in df.columns:
    df['yield_curve_cycle'] = df.groupby('country')['yield_curve_slope'].transform(apply_hp_filter)

# ---------------------------------------------------------
# 4. FEATURE SELECTION & CLEANING
# ---------------------------------------------------------
target_col = 'crisisJST'

desired_features = [
    'yield_curve_slope', 
    'credit_gdp', 
    'credit_gdp_diff2', 
    'credit_gdp_cycle', 
    'yield_curve_cycle', 
    'cpi', 
    'unemp', 
    'debtgdp'
]

available_features = [f for f in desired_features if f in df.columns]
print(f"Active features extracted for modeling ({len(available_features)}/8): {available_features}")

df_processed = df.dropna(subset=[target_col]).copy()
df_processed[available_features] = df_processed[available_features].fillna(0)

# Display active features as a clean table in Jupyter
df_summary = pd.DataFrame({
    "Feature Name": available_features,
    "Status": ["Active"] * len(available_features),
    "Source / Processing": [
        "HP-Filtered" if "cycle" in col else "Engineered/Raw" for col in available_features
    ]
})
display(df_summary)

# ---------------------------------------------------------
# 5. EXPORT PROCESSED SIGNALS
# ---------------------------------------------------------
output_path = 'data/processed_signals.csv'
df_processed.to_csv(output_path, index=False)

print(f"\n✅ Preprocessing Complete! Saved to '{output_path}'.")
print(f"Total rows ready for Hybrid Model training: {len(df_processed)}")

=== FINSIGHT AI: STEP 2 - REAL-WORLD PREPROCESSING & HP FILTER ===

Successfully loaded data: 2668 rows.
Applying the winning HP Filter grouped by country...
Active features extracted for modeling (8/8): ['yield_curve_slope', 'credit_gdp', 'credit_gdp_diff2', 'credit_gdp_cycle', 'yield_curve_cycle', 'cpi', 'unemp', 'debtgdp']


,Feature Name,Status,Source / Processing
0,yield_curve_slope,Active,Engineered/Raw
1,credit_gdp,Active,Engineered/Raw
2,credit_gdp_diff2,Active,Engineered/Raw
3,credit_gdp_cycle,Active,HP-Filtered
4,yield_curve_cycle,Active,HP-Filtered
5,cpi,Active,Engineered/Raw
6,unemp,Active,Engineered/Raw
7,debtgdp,Active,Engineered/Raw



✅ Preprocessing Complete! Saved to 'data/processed_signals.csv'.
Total rows ready for Hybrid Model training: 2668


In [12]:
print(pd.read_csv('data/01_cleaned_raw.csv').columns.tolist())

['year', 'country', 'iso', 'ifs', 'pop', 'rgdpmad', 'rgdpbarro', 'rconsbarro', 'gdp', 'iy', 'cpi', 'ca', 'imports', 'exports', 'narrowm', 'money', 'stir', 'ltrate', 'hpnom', 'unemp', 'wage', 'debtgdp', 'revenue', 'expenditure', 'xrusd', 'tloans', 'tmort', 'thh', 'tbus', 'bdebt', 'lev', 'ltd', 'noncore', 'crisisJST', 'crisisJST_old', 'peg', 'peg_strict', 'peg_type', 'peg_base', 'JSTtrilemmaIV', 'eq_tr', 'housing_tr', 'bond_tr', 'bill_rate', 'rent_ipolated', 'housing_capgain_ipolated', 'housing_capgain', 'housing_rent_rtn', 'housing_rent_yd', 'eq_capgain', 'eq_dp', 'eq_capgain_interp', 'eq_tr_interp', 'eq_dp_interp', 'bond_rate', 'eq_div_rtn', 'capital_tr', 'risky_tr', 'safe_tr']
